# Transaction types

One example row per type of securities lending transaction, from the cleaned table on the latest reference period.

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

Which types are there and how frequent are they?

In [ ]:
query = f"""

SELECT collateral_type, COUNT(*) AS n
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
GROUP BY 1
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

One example per type. Key, when, who, what.

In [ ]:
query = f"""

SELECT uti, reference_period, start_date, lender_id, borrower_id, agent_lender_id,
       isin, loan_quantity, loan_value_eur, collateral_type, lending_fee, rebate_rate
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY collateral_type ORDER BY uti) AS rn
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
) x
WHERE rn = 1
ORDER BY collateral_type

"""
df = pd.read_sql_query(query, cnxn)
df

The collateral pieces of the examples that have any.

In [ ]:
utis = "', '".join(df['uti'])
query = f"""

SELECT l.uti, c.collateral_kind, c.collateral_isin, c.collateral_quantity, c.cash_amount, c.cash_currency, c.haircut
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl l
  ON l.tec_ruti = c.tec_ruti AND l.reference_period = c.reference_period
WHERE l.reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
  AND l.uti IN ('{utis}')
ORDER BY l.uti, c.collateral_index

"""
df = pd.read_sql_query(query, cnxn)
df